# 🎯 Topic Mastery Suite
### Master *any* topic you just learned — entirely inside this notebook.

**How it works:**
1. **Run the setup cells** (connects to your LLM provider automatically)
2. **Enter the topic you just learned** — the AI identifies the subject/language, and asks you to clarify if the topic is too broad
3. **Run `new_question()`** — a fresh, never-repeated question appears, along with an **empty answer cell** below it
4. **Write your answer in that cell and run it** — the AI instantly grades it: ✅ correct / 🟡 partially correct / ❌ incorrect, **plus feedback and a model answer**

> Nothing here is hardcoded. Every question is generated live by the LLM, a history file guarantees no repetition (even across sessions), and every evaluation is done by the AI — no deterministic answer keys.

---

## ⚙️ Step 0 — Setup
Run the two cells below once. The engine auto-detects your LLM provider:
`OPENROUTER_API_KEY` → `GEMINI_API_KEY` → `OPENAI_API_KEY` → local **Ollama**.

*(Put your key in a `.env` file next to this notebook — see `.env.example`.)*

In [1]:
# Install the only dependencies (safe to re-run)
%pip install -q requests python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from topic_mastery_engine import MasteryEngine, LLMError
from IPython.display import display, Markdown

try:
    ENGINE = MasteryEngine()
    print(f"✅ Connected to provider: {ENGINE.client.provider}  |  model: {ENGINE.client.model}")
except LLMError as e:
    print(f"❌ {e}")
    print("👉 Add an API key to a .env file (see .env.example), then re-run this cell.")

✅ Connected to provider: openrouter  |  model: openrouter/free


## 📚 Step 1 — Enter the topic you just learned
Type anything — *"Python decorators"*, *"the Krebs cycle"*, *"French Revolution"*, *"React hooks"*...

If the AI finds your topic too broad (e.g. just *"science"*), it will **ask you clarifying questions** before continuing.

In [3]:
topic = input("📚 What topic did you just learn? ").strip()

while True:
    analysis = ENGINE.analyze_topic(topic)
    if not analysis.get("is_ambiguous"):
        break
    print("\n🤔 That topic is a bit broad — the AI needs clarification:")
    for i, q in enumerate(analysis["clarifying_questions"], 1):
        print(f"   {i}. {q}")
    topic = f"{topic} — {input('Your clarification: ').strip()}"

ENGINE.set_topic(topic, analysis)

display(Markdown(f"""
### ✅ Topic locked in
| | |
|---|---|
| **Topic** | {analysis['refined_topic']} |
| **Subject / Language** | {analysis['subject']} |
| **Scope** | {analysis['scope_summary']} |
"""))


### ✅ Topic locked in
| | |
|---|---|
| **Topic** | Arrow function syntax and behavior in JavaScript |
| **Subject / Language** | JavaScript programming |
| **Scope** | A quiz on this topic should cover arrow function syntax, lexical this binding, differences from regular functions, and common use cases or limitations. |


## 📝 Step 2 — Practice loop
Run the cell below to define the helpers (once), then use:
- **`new_question()`** → generates a unique question and opens an **empty answer cell** below — write your answer in it and run it
- **`hint()`** → get a Socratic nudge for the current question
- **`new_question()`** again → next fresh question (never a repeat)
- **`session_report()`** → your scorecard

In [4]:
def new_question(difficulty="auto"):
    """Generate a fresh question and open an empty answer cell below."""
    q = ENGINE.ask_question(difficulty=difficulty)
    n = len(ENGINE._past_questions())
    display(Markdown(f"""
---
### 📝 Question {n} &nbsp; `{q['type']}` · `{q['difficulty']}`

{q['question']}

> ✍️ **Write your answer in the cell below, then run it.** *(Stuck? Run `hint()` in a new cell.)*
"""))
    # Creates the empty answer cell right under this one
    get_ipython().set_next_input("my_answer = " + chr(34)*3 + "\n\n" + chr(34)*3 + "\n\ncheck_answer(my_answer)", replace=False)


def check_answer(answer):
    """Grade the answer in the current cell with the AI."""
    if not answer.strip():
        print("✏️ Write your answer inside the triple quotes first, then run this cell.")
        return
    r = ENGINE.evaluate(answer)
    icon = {"correct": "✅ CORRECT", "partially_correct": "🟡 PARTIALLY CORRECT", "incorrect": "❌ INCORRECT"}[r["verdict"]]
    missed = "\n".join(f"- {p}" for p in r.get("key_points_missed", [])) or "- None 🎉"
    display(Markdown(f"""
---
## {icon} &nbsp; — &nbsp; Score: **{r['score']}/10**

**💬 Feedback:** {r['feedback']}

**📖 Model answer:**
{r['model_answer']}

**🔍 Key points missed:**
{missed}

---
*Run `new_question()` for the next one.*"""))


def hint():
    display(Markdown(f"💡 **Hint:** {ENGINE.get_hint()}"))


def session_report():
    s = ENGINE.summary()
    if not s["attempted"]:
        print("No answers graded yet.")
        return
    display(Markdown(f"""
## 📊 Session Report — {ENGINE.topic}
| | |
|---|---|
| Questions attempted | {s['attempted']} |
| ✅ Correct | {s['correct']} |
| 🟡 Partially correct | {s['partially_correct']} |
| ❌ Incorrect | {s['incorrect']} |
| **Average score** | **{s['average_score']}/10** |
"""))

print("✅ Helpers ready. Now run:  new_question()")

✅ Helpers ready. Now run:  new_question()


### ▶️ Start practicing — run the cell below:

In [5]:
new_question()


---
### 📝 Question 1 &nbsp; `conceptual` · `medium`

In JavaScript, arrow functions exhibit distinct `this` binding behavior compared to regular function expressions. When you attach an arrow function directly to a DOM element's event handler or pass it as a callback to higher-order functions, it captures the `this` bound from its enclosing lexical scope rather than creating its own isolated `this`. Consider the following code:

javascript
authButton.addEventListener('submit', () => {
  console.log(this.name); // What gets logged?
});


If `authButton` refers to a form element in a React component tree without proper context propagation, and the inner function shadows `this` incorrectly due to closure mechanics, explain the consequence. Then compare this outcome with a traditional function expression `handleSubmit = () => console.log(this.name);` declared outside the element, noting how each would resolve the same `this` reference. Finally, describe a valid pattern for sharing custom `this` binding values across many arrow functions.

Answer should address both the conceptual mechanism and practical implications.

> ✍️ **Write your answer in the cell below, then run it.** *(Stuck? Run `hint()` in a new cell.)*


In [6]:
my_answer = """
the pattern for sharing custom this is by using the .bind , .call or .apply method for pass the proper context 
"""

check_answer(my_answer)

LLMError: LLM did not return JSON:
User Safety: safe

---
*Tip: every question you answer is remembered in `.topic_mastery_history.json`, so you'll never see the same question twice — even in a future session. Delete that file to reset.*